# Stage 3 Forced Matching Audit

**Objective:** verify that the 173 Stage 2 holds are force-matched to their recorded hierarchy-blind top-1 L3 while all 1,553 Stage 2 placements remain unchanged.

**Authority boundary:** 100% coverage is an operational mapping, not human-approved taxonomy ground truth. No charts or images are produced.

In [1]:
from pathlib import Path
import json
from collections import Counter

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    ROOT = ROOT.parents[1]
STAGE2 = ROOT / 'data/experiments/stage2-v1'
STAGE3 = ROOT / 'data/experiments/stage3-v1'
BASE = ROOT / 'data/releases/v1.0.0'
VALIDATION = ROOT / 'reports/validation/stage3-v1'

def load(path):
    return json.loads(path.read_text(encoding='utf-8'))

stage2 = load(STAGE2 / 'stage2_placements.json')
stage3 = load(STAGE3 / 'stage3_placements.json')
scores = load(BASE / 'algorithm_scores.json')
summary = load(STAGE3 / 'stage3_summary.json')
validation = load(VALIDATION / 'validation_summary.json')
print({'stage2_rows': len(stage2), 'stage3_rows': len(stage3)})

{'stage2_rows': 1726, 'stage3_rows': 1726}


## Planned checks

1. Every L4 has exactly one non-null Stage 3 L3.
2. Exactly 173 former Stage 2 holds are force-matched.
3. All prior Physical, Stage 1, and Stage 2 placements are preserved.
4. Every forced assignment equals its recorded hierarchy-blind top-1 L3.
5. Hold reasons and explicit non-approval flags remain available.

In [2]:
stage2_by_id = {row['l4_id']: row for row in stage2}
stage3_by_id = {row['l4_id']: row for row in stage3}
score_by_id = {row['l4_id']: row for row in scores}
forced = [row for row in stage3 if row['forced_match']]
checks = {
    'rows_1726': len(stage3) == 1726,
    'unique_ids_1726': len(stage3_by_id) == 1726,
    'all_l3_non_null': all(row['stage3_l3_id'] for row in stage3),
    'forced_173': len(forced) == 173,
    'forced_were_stage2_holds': all(stage2_by_id[row['l4_id']]['stage2_status'] == 'needs_taxonomy_decision' for row in forced),
    'forced_equals_top1': all(row['stage3_l3_id'] == score_by_id[row['l4_id']]['top1_l3_id'] for row in forced),
}
print(checks)
assert all(checks.values())

{'rows_1726': True, 'unique_ids_1726': True, 'all_l3_non_null': True, 'forced_173': True, 'forced_were_stage2_holds': True, 'forced_equals_top1': True}


In [3]:
preserved = [row for row in stage2 if row['stage2_l3_id'] is not None]
preservation_failures = [row['l4_id'] for row in preserved if stage3_by_id[row['l4_id']]['stage3_l3_id'] != row['stage2_l3_id']]
print({'stage2_placements_checked': len(preserved), 'preservation_failures': len(preservation_failures)})
assert not preservation_failures

{'stage2_placements_checked': 1553, 'preservation_failures': 0}


In [4]:
forced_profile = {
    'forced_total': len(forced),
    'hard_hold_forced': sum(row['forced_hold_class'] == 'hard_hold' for row in forced),
    'quota_hold_forced': sum(row['forced_hold_class'] == 'quota_hold' for row in forced),
    'critical_review': sum(row['stage3_review_priority'] == 'critical' for row in forced),
    'high_review': sum(row['stage3_review_priority'] == 'high' for row in forced),
    'hold_reason_preserved': all(bool(row['forced_from_stage2_hold_reason']) for row in forced),
}
print(forced_profile)

{'forced_total': 173, 'hard_hold_forced': 55, 'quota_hold_forced': 118, 'critical_review': 55, 'high_review': 118, 'hold_reason_preserved': True}


In [5]:
status_counts = Counter(row['stage3_status'] for row in stage3)
l3_counts = Counter(row['stage3_l3_id'] for row in stage3)
print({'status_counts': status_counts, 'top_five_l3': l3_counts.most_common(5), 'largest_share': summary['largest_l3_share']})
assert summary['classified_rate'] == 1.0
assert summary['largest_l3_share'] < 0.20

{'status_counts': Counter({'algorithm_proposed_stage2': 1334, 'locked_physical': 182, 'forced_match_stage3': 173, 'stage1_algorithm_proposed': 37}), 'top_five_l3': [('RAI3-G-INT-10', 254), ('RAI3-G-SYS-08', 153), ('RAI3-G-INT-08', 118), ('RAI3-G-SYS-03', 110), ('RAI3-G-INT-06', 90)], 'largest_share': 0.147161}


In [6]:
authority_checks = {
    'no_human_approval_claim': not any(row['human_approved'] for row in stage3),
    'forced_uncalibrated': all(row['confidence_calibrated'] is False for row in forced),
    'legacy_hierarchy_excluded': not any(row['legacy_hierarchy_used_as_feature'] for row in stage3),
    'validator_passed': validation['status'] == 'PASS_WITH_WARNINGS' and validation['failed_checks'] == 0,
}
print(authority_checks)
assert all(authority_checks.values())

{'no_human_approval_claim': True, 'forced_uncalibrated': True, 'legacy_hierarchy_excluded': True, 'validator_passed': True}


## Result

Stage 3 provides a complete operational path for all 1,726 cards. It does not convert the 173 former holds into validated truth: their forced status, original hold reason, review priority, uncalibrated flag, and non-approval status remain explicit. The 55 forced hard holds are the highest-priority review set.